### Import libraries

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import emoji
import nltk

In [32]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# <div style="text-align: center; background-color: white; font-family: Lobster;color: black; padding: 14px; line-height: 1;border-radius:12px"> Pre-processing "Posts.csv" </div>

In [33]:
posts_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\clean\clean_posts.csv')
posts_df.head()

,post_id,timestamp,postUser,caption,hashtags,likesCount,commentsCount
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",no_hashtag,6565,86
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,no_hashtag,28936,123
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,CHANELFallWinter,4764,215
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,CannesFilmFestival,13379,123


In [34]:
posts_df.isnull().sum()

post_id          0
timestamp        0
postUser         0
caption          0
hashtags         0
likesCount       0
commentsCount    0
dtype: int64

In [35]:
# Type of each column
posts_df['timestamp'] = pd.to_datetime(posts_df['timestamp'])
posts_df.dtypes   

post_id                        int64
timestamp        datetime64[ns, UTC]
postUser                      object
caption                       object
hashtags                      object
likesCount                     int64
commentsCount                  int64
dtype: object

### "Caption" column 

In [36]:
def preprocess_caption(text):
    if pd.isna(text) or text == 'no_caption':
        return ""
    
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    
    # Keep alphanumeric, whitespace, and emoji characters, replace others with space
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip() 
    
    return text

posts_df['clean_caption'] = posts_df['caption'].apply(preprocess_caption)

In [37]:
# Process Emoji in caption

def process_emoji(text):
    """
    Process emojis in text.
    Returns text without emojis and list of emoji descriptions.
    """
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', ' ')
        emoji_descriptions.append(emoji_name)
    
    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

posts_df[['caption_no_emoji', 'emoji_descriptions']] = posts_df['clean_caption'].apply(lambda x: pd.Series(process_emoji(x)))


In [38]:
posts_df.head()

,post_id,timestamp,postUser,caption,hashtags,likesCount,commentsCount,clean_caption,caption_no_emoji,emoji_descriptions
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142,cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart]
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",no_hashtag,6565,86,as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu..."
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,no_hashtag,28936,123,the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart]
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,CHANELFallWinter,4764,215,i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]"
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,CannesFilmFestival,13379,123,got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart]


### "Hashtags" column

In [39]:
# Process hashtags from hashtag column
def process_hashtags_from_column(hashtag_text):
    if pd.isna(hashtag_text) or hashtag_text == 'no_hashtag' or hashtag_text == '':
        return []
    
    if hashtag_text.startswith('[') and hashtag_text.endswith(']'):
        hashtag_text = hashtag_text[1:-1]
        
        tags = [tag.strip().strip("'").strip('"') for tag in hashtag_text.split(',')]
    else:
        tags = [tag.strip().strip('#') for tag in re.split(r'[,#]', hashtag_text) if tag.strip()]
    
    processed_hashtags = []
    for tag in tags:
        if tag:
            words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
            if not words:
                words = [tag]
            processed_hashtags.extend([word.lower() for word in words])
    
    return processed_hashtags

posts_df['hashtags'] = posts_df['hashtags'].apply(process_hashtags_from_column)

In [41]:
# Tokenize and remove stopwords
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def tokenize_and_remove_stopwords(text):
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

posts_df['caption_tokens'] = posts_df['caption_no_emoji'].apply(tokenize_and_remove_stopwords)

In [44]:
posts_df['combined_tokens'] = posts_df.apply(lambda row: row['caption_tokens'] + row['emoji_descriptions'] + row['hashtags'], axis=1)

In [45]:
processed_posts_df = posts_df[['post_id', 'postUser', 'combined_tokens']]
processed_posts_df.head()

,post_id,postUser,combined_tokens
0,1,maryleest,"[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,tinaabeysekara,"[clock, struck, midnight, ring, 2022, cried, w..."
2,3,maryleest,"[famous, stairs, photo, gustave_durin, dress, ..."
3,4,stephaniebroek,"[visualized, moment, many, times, first, chane..."
4,5,maryleest,"[got, witness, historical, moment, cinema, kil..."


# <div style="text-align: center; background-color: white; font-family: Lobster;color: black; padding: 14px; line-height: 1;border-radius:12px"> Pre-processing "Comments.csv" </div>